In [1]:
!pip install ultralytics opencv-python-headless pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.6 MB/s eta 0:00:00


In [ ]:
import cv2
import torch
import numpy as np
import time
from ultralytics import YOLO
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode
import PIL

# ------------------------------------------------------------
# 1. JavaScript helper to capture a single frame from webcam
# ------------------------------------------------------------
def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capture';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            // Resize the output to fit the video element.
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            // Wait for Capture to be clicked.
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
        ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    return np.frombuffer(binary, dtype=np.uint8)

# ------------------------------------------------------------
# 2. Real-time detection class for Colab (polling mode)
# ------------------------------------------------------------
class RealtimeYOLODetectionColab:
    def __init__(self, model_path='yolov5su.pt'):
        self.model = YOLO(model_path)
        self.colors = np.random.randint(0, 255, size=(80, 3), dtype=np.uint8)

    def run(self):
        print("📸 Press the 'Capture' button in the browser window to grab a frame.")
        print("💡 To exit, interrupt the cell (Runtime → Interrupt execution).")

        while True:
            # Capture a frame from the webcam (blocking until button press)
            img_bytes = take_photo()
            frame = cv2.imdecode(img_bytes, cv2.IMREAD_COLOR)

            if frame is None:
                print("No frame captured. Exiting.")
                break

            start_time = time.time()

            # Perform inference
            results = self.model(frame)

            # Process results
            for result in results:
                boxes = result.boxes.cpu().numpy()
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].astype(int)
                    cls = int(box.cls[0])
                    conf = float(box.conf[0])
                    label = f"{result.names[cls]} {conf:.2f}"
                    color = [int(c) for c in self.colors[cls]]

                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame, label, (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Calculate FPS (approximate)
            fps = 1.0 / (time.time() - start_time)
            cv2.putText(frame, f"FPS: {fps:.2f}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # Display the frame in the notebook
            # Convert BGR to RGB for correct display
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = PIL.Image.fromarray(frame_rgb)
            display(img)   # shows each frame one after another

            # Optional: small delay to avoid overloading the notebook
            time.sleep(0.1)

# ------------------------------------------------------------
# 3. Usage
# ------------------------------------------------------------
if __name__ == "__main__":
    detector = RealtimeYOLODetectionColab()
    detector.run()

In [8]:
# 1. Install dependencies (if not already installed)
!pip install ultralytics opencv-python-headless pillow -q

In [ ]:


import cv2
import numpy as np
import time
import json
import torch
from ultralytics import YOLO
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode

# ------------------------------------------------------------
# CONFIGURATION (adjust as needed)
# ------------------------------------------------------------
MODEL_PATH = "yolov8n.pt"      # or custom path
CONF = 0.4
IOU = 0.45
IMGSZ = 320                    # auto‑adjusted: 256 for CPU, 320+ for GPU
JPEG_QUALITY = 0.6
MAX_DET = 100
TARGET_FPS = 0.5               # low FPS for CPU; adjust for your runtime

# ------------------------------------------------------------
# 2. Detect device (CPU/GPU)
# ------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device.upper()}")

# Adjust IMGSZ for CPU to be smaller
if device == "cpu" and IMGSZ > 256:
    IMGSZ = 256
    print(f"CPU detected, setting IMGSZ to {IMGSZ} for performance.")

# ------------------------------------------------------------
# 3. Load YOLO model
# ------------------------------------------------------------
model = YOLO(MODEL_PATH).to(device)
print(f"Model loaded on {device}")

# ------------------------------------------------------------
# 4. Inject JavaScript: Camera, Canvas, and Rendering
# ------------------------------------------------------------
js_code = """
// ---------- Global state ----------
window.latestFrame = null;          // Base64 JPEG of the latest video frame
window.detections = [];             // Latest detections (JSON array)
window.video = null;
window.stream = null;
window.canvas = null;
window.ctx = null;
window.isRunning = false;
window.lastDrawTime = 0;

// ---------- Camera & capture loop ----------
async function initCamera() {
    try {
        window.stream = await navigator.mediaDevices.getUserMedia({ video: true });
        window.video = document.createElement('video');
        window.video.style.display = 'none';
        document.body.appendChild(window.video);
        window.video.srcObject = window.stream;
        await window.video.play();

        // Create and display canvas
        window.canvas = document.createElement('canvas');
        window.canvas.width = 640;
        window.canvas.height = 480;
        window.canvas.style.display = 'block';
        window.canvas.style.margin = 'auto';
        window.ctx = window.canvas.getContext('2d');
        document.body.appendChild(window.canvas);

        window.isRunning = true;
        console.log('Camera ready');
        captureLoop();
    } catch (e) {
        console.error('Camera init error:', e);
    }
}

// Capture frames continuously using requestAnimationFrame
function captureLoop() {
    if (!window.isRunning) return;
    if (window.video && window.video.readyState >= 2) {
        // Capture frame to canvas (off‑screen) and encode as JPEG
        const offCanvas = document.createElement('canvas');
        offCanvas.width = 640;   // resize before encoding to reduce transfer size
        offCanvas.height = 480;
        const offCtx = offCanvas.getContext('2d');
        offCtx.drawImage(window.video, 0, 0, 640, 480);
        window.latestFrame = offCanvas.toDataURL('image/jpeg', 0.6);
    }
    requestAnimationFrame(captureLoop);
}

// ---------- Draw detections on the canvas ----------
function drawFrame() {
    if (!window.canvas || !window.ctx) return;
    const ctx = window.ctx;
    const canvas = window.canvas;

    // If we have a latest frame, draw it
    if (window.latestFrame) {
        const img = new Image();
        img.onload = function() {
            ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
            drawDetections();    // overlay detections
        };
        img.src = window.latestFrame;
    } else {
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        ctx.fillStyle = 'black';
        ctx.fillRect(0, 0, canvas.width, canvas.height);
        ctx.fillStyle = 'white';
        ctx.font = '20px Arial';
        ctx.fillText('Waiting for camera...', 10, 30);
    }
}

function drawDetections() {
    const ctx = window.ctx;
    const canvas = window.canvas;
    const dets = window.detections || [];

    ctx.font = '14px Arial';
    dets.forEach(d => {
        const x1 = d.x1, y1 = d.y1, x2 = d.x2, y2 = d.y2;
        const label = `${d.class} ${(d.conf*100).toFixed(1)}%`;
        const color = `hsl(${(d.class_id * 40) % 360}, 70%, 50%)`;

        // Draw box
        ctx.strokeStyle = color;
        ctx.lineWidth = 2;
        ctx.strokeRect(x1, y1, x2-x1, y2-y1);

        // Draw label background
        ctx.fillStyle = color;
        const textWidth = ctx.measureText(label).width;
        ctx.fillRect(x1, y1 - 20, textWidth + 8, 20);

        // Draw label text
        ctx.fillStyle = 'white';
        ctx.fillText(label, x1 + 4, y1 - 4);
    });
}

// Update detections from Python
function updateDetections(data) {
    window.detections = data;
    // drawFrame will be called by the animation loop
    drawFrame();   // immediate redraw
}

// ---------- Start everything ----------
initCamera();
// Periodic redraw (if not triggered by updates)
setInterval(drawFrame, 100);
"""

display(Javascript(js_code))

# Wait a moment for the camera to initialize
time.sleep(2)

# ------------------------------------------------------------
# 5. Python inference loop (low FPS)
# ------------------------------------------------------------
print("\nStarting detection loop... Press Stop button to interrupt.\n")

frame_count = 0
while True:
    loop_start = time.time()

    # --- 1. Get latest frame from JavaScript (lightweight read) ---
    try:
        img_data = eval_js('window.latestFrame')
        if not img_data:
            print("Waiting for first frame...")
            time.sleep(1)
            continue
    except Exception as e:
        print(f"Error reading frame: {e}")
        break

    # --- 2. Decode JPEG to OpenCV image ---
    binary = b64decode(img_data.split(',')[1])
    frame = cv2.imdecode(np.frombuffer(binary, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        print("Failed to decode frame")
        time.sleep(0.5)
        continue

    # --- 3. Run YOLO inference ---
    results = model(frame, imgsz=IMGSZ, conf=CONF, iou=IOU, max_det=MAX_DET, verbose=False)

    # --- 4. Extract detections as JSON-friendly list ---
    detections = []
    for result in results:
        boxes = result.boxes.cpu().numpy()
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].astype(int)
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            detections.append({
                "x1": int(x1),
                "y1": int(y1),
                "x2": int(x2),
                "y2": int(y2),
                "class": result.names[cls],
                "class_id": cls,
                "conf": conf
            })

    # --- 5. Send detections back to browser (lightweight JSON) ---
    try:
        eval_js(f'updateDetections({json.dumps(detections)})')
    except Exception as e:
        print(f"Error sending detections: {e}")

    # --- 6. Show performance info in Python output (optional, not in canvas) ---
    elapsed = time.time() - loop_start
    fps = 1.0 / elapsed if elapsed > 0 else 0
    print(f"Frame {frame_count}: {len(detections)} objects, inference time {elapsed:.3f}s, FPS {fps:.2f}", end='\r')
    frame_count += 1

    # --- 7. Enforce target FPS (sleep) ---
    sleep_time = (1.0 / TARGET_FPS) - elapsed
    if sleep_time > 0:
        time.sleep(sleep_time)

    # Optional: break on Notebook interrupt (handled by KeyboardInterrupt)